In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [ ]:
import numpy as np
from loaders._gen_binary import generate_data
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import balanced_accuracy_score

# Data Generation

In [3]:
_ = generate_data(verbose=True)

Generated dataset with 1995 samples.
Shapes: [(1596, 25), (0, 25), (399, 25)]
Label distribution: Counter({np.int64(1): 817, np.int64(0): 779})
[WARNING] If you use a tree-based model, consider setting use_scaler=False.


# SVM (Original)

In [4]:
bacc = []

for seed in range(30):
    pack = generate_data(seed=seed)
    X_train, y_train = pack["train"]
    X_test, y_test = pack["test"]

    tscv = TimeSeriesSplit(n_splits=5)

    params = {
        "C": [0.001, 0.01, 0.1, 0.3, 1, 3, 10],
        "kernel": ["linear", "rbf", "poly", "sigmoid"],
        "gamma": ["scale", "auto"],
        "degree": [2, 3, 4],
    }

    search = GridSearchCV(
        estimator=SVC(),
        param_grid=params,
        cv=tscv,
        n_jobs=-1,
    )

    search.fit(X_train, y_train)
    y_preds = search.predict(X_test)

    bacc.append(balanced_accuracy_score(y_test, y_preds))

    print(f"Seed {seed} - Test Balanced Accuracy: {bacc[-1]}")

print(f"Mean Test Balanced Accuracy: {np.mean(bacc)}")
print(f"Std Test Balanced Accuracy: {np.std(bacc)}")

Seed 0 - Test Balanced Accuracy: 0.6867211055276382
Seed 1 - Test Balanced Accuracy: 0.6765482054890922
Seed 2 - Test Balanced Accuracy: 0.634554300695775
Seed 3 - Test Balanced Accuracy: 0.6523626207729469


KeyboardInterrupt: 